In [20]:
import pandas as pd
import re

# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv('XCM.csv')

# ----------------------------
# Function to parse a discount card string
# ----------------------------
def parse_card(card_text):
    """
    Parse a single discount card text like:
    'Item_D on DB: 25% off, now €112.5'
    Returns item, train, discount, discounted_price
    """
    match = re.match(r'(\w+)\s+on\s+(\w+):\s+(\d+)% off, now €([\d.]+)', card_text)
    if match:
        item, train, discount, price = match.groups()
        return item, train, int(discount), float(price)
    else:
        return None

# ----------------------------
# Split list into separate rows
# ----------------------------
rows = []
for idx, row in df.iterrows():
    date = row['Date']
    discount_text = row['Discount_Cards']
    
    if discount_text == "No discounts":
        continue  # skip days with no discounts
    else:
        # Remove brackets and split by comma (taking care of commas in numbers)
        card_texts = re.findall(r'[^,]+on\s\w+:\s\d+% off, now €[\d.]+', discount_text)
        for card in card_texts:
            parsed = parse_card(card.strip())
            if parsed:
                item, train, discount, discounted_price = parsed
                rows.append({
                    "Date": date,
                    "Item": item,
                    "Train": train,
                    "Discount_%": discount,
                    "Discounted_Price": discounted_price
                })

# ----------------------------
# Create DataFrame
# ----------------------------
df_expanded = pd.DataFrame(rows)

# Save to CSV
df_expanded.to_csv('XCM_expanded.csv', index=False)

print("CSV file 'XCM_expanded.csv' created with individual rows for each discount!")

CSV file 'XCM_expanded.csv' created with individual rows for each discount!


In [ ]:
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, mean_squared_error
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv('XCM_expanded.csv')  # Date, Item, Train, Discount_%, Discounted_Price

# ----------------------------
# Convert Date to datetime & create features
# ----------------------------
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df = df.sort_values('Date')

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.weekday
df['IsWeekend'] = (df['Weekday'] >= 5).astype(int)
df['Season'] = ((df['Month'] % 12 + 3) // 3 - 1)
df['Month_sin'] = np.sin(2*np.pi*df['Month']/12)
df['Month_cos'] = np.cos(2*np.pi*df['Month']/12)

# Lag features to help prediction (boost accuracy)
df['Discount_Lag1'] = df.groupby(['Item','Train'])['Discount_%'].shift(1).fillna(0)
df['Discount_Lag7'] = df.groupby(['Item','Train'])['Discount_%'].shift(7).fillna(0)
df['Price_Lag1'] = df.groupby(['Item','Train'])['Discounted_Price'].shift(1).fillna(df['Discounted_Price'].mean())
df['Price_Lag7'] = df.groupby(['Item','Train'])['Discounted_Price'].shift(7).fillna(df['Discounted_Price'].mean())

# ----------------------------
# Group by Date for multi-label
# ----------------------------
df_grouped = df.groupby('Date').agg({
    'Item': lambda x: list(x),
    'Train': lambda x: list(x),
    'Discount_%': lambda x: list(x),
    'Discounted_Price': lambda x: list(x)
}).reset_index()

# ----------------------------
# Features
# ----------------------------
X = pd.DataFrame({
    'Year': df_grouped['Date'].dt.year,
    'Month': df_grouped['Date'].dt.month,
    'Day': df_grouped['Date'].dt.day,
    'Weekday': df_grouped['Date'].dt.weekday,
    'IsWeekend': (df_grouped['Date'].dt.weekday >= 5).astype(int),
    'Season': ((df_grouped['Date'].dt.month % 12 + 3)//3 -1),
    'Month_sin': np.sin(2*np.pi*df_grouped['Date'].dt.month/12),
    'Month_cos': np.cos(2*np.pi*df_grouped['Date'].dt.month/12)
})

# ----------------------------
# Encode Item and Train for multi-label
# ----------------------------
mlb_item = MultiLabelBinarizer()
mlb_train = MultiLabelBinarizer()
Y_item = mlb_item.fit_transform(df_grouped['Item'])
Y_train = mlb_train.fit_transform(df_grouped['Train'])

# ----------------------------
# Discount & Price as padded arrays
# ----------------------------
max_len_discount = max(df_grouped['Discount_%'].apply(len))
Y_discount = np.array([d + [0]*(max_len_discount - len(d)) for d in df_grouped['Discount_%']])
max_len_price = max(df_grouped['Discounted_Price'].apply(len))
Y_price = np.array([p + [0]*(max_len_price - len(p)) for p in df_grouped['Discounted_Price']])

# ----------------------------
# Train/test split
# ----------------------------
X_train, X_test, Y_item_train, Y_item_test, Y_train_train, Y_train_test, Y_discount_train, Y_discount_test, Y_price_train, Y_price_test = train_test_split(
    X, Y_item, Y_train, Y_discount, Y_price, test_size=0.2, random_state=42
)

# ----------------------------
# XGBoost MultiOutput models
# ----------------------------
xgb_item = MultiOutputClassifier(CatBoostClassifier(iterations=400, depth=6, learning_rate=0.1, verbose=0))
xgb_item.fit(X_train, Y_item_train)

xgb_train = MultiOutputClassifier(CatBoostClassifier(iterations=400, depth=6, learning_rate=0.1, verbose=0))
xgb_train.fit(X_train, Y_train_train)

xgb_reg_discount = MultiOutputRegressor(XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                                                     colsample_bytree=0.8, random_state=42))
xgb_reg_discount.fit(X_train, Y_discount_train)

xgb_reg_price = MultiOutputRegressor(XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                                                  colsample_bytree=0.8, random_state=42))
xgb_reg_price.fit(X_train, Y_price_train)

# ----------------------------
# Predictions
# ----------------------------
Y_item_pred = xgb_item.predict(X_test)
Y_train_pred = xgb_train.predict(X_test)
Y_discount_pred = xgb_reg_discount.predict(X_test)
Y_price_pred = xgb_reg_price.predict(X_test)

# ----------------------------
# Evaluation
# ----------------------------
item_acc = [accuracy_score(Y_item_test[:,i], Y_item_pred[:,i]) for i in range(Y_item_test.shape[1])]
train_acc = [accuracy_score(Y_train_test[:,i], Y_train_pred[:,i]) for i in range(Y_train_test.shape[1])]
discount_rmse = np.sqrt(mean_squared_error(Y_discount_test.flatten(), Y_discount_pred.flatten()))
price_rmse = np.sqrt(mean_squared_error(Y_price_test.flatten(), Y_price_pred.flatten()))

print("Item-wise accuracy:", item_acc)
print("Train-wise accuracy:", train_acc)
print("Discount % RMSE:", discount_rmse)
print("Discounted Price RMSE:", price_rmse)

# ----------------------------
# Example prediction for a new date
# ----------------------------
example_date = pd.DataFrame({
    'Year':[2020],
    'Month':[6],
    'Day':[14],
    'Weekday':[datetime.datetime(2020,6,14).weekday()],
    'IsWeekend':[1 if datetime.datetime(2020,6,14).weekday()>=5 else 0],
    'Season':[ (6%12 +3)//3 -1 ],
    'Month_sin':[np.sin(2*np.pi*6/12)],
    'Month_cos':[np.cos(2*np.pi*6/12)]
})

pred_items = mlb_item.inverse_transform(xgb_item.predict(example_date))
pred_trains = mlb_train.inverse_transform(xgb_train.predict(example_date))
pred_discounts = xgb_reg_discount.predict(example_date)
pred_prices = xgb_reg_price.predict(example_date)

if len(pred_items[0]) == 0 or len(pred_trains[0]) == 0:
    print("No discounts predicted for this date.")
else:
    discount_list = [f"{item} on {train}: {discount:.0f}% off, now €{price:.2f}"
                     for item, train, discount, price in zip(pred_items[0], pred_trains[0], pred_discounts[0], pred_prices[0])]
    print("\nPrediction for 14-06-2020:")
    for d in discount_list:
        print(d)

Item-wise accuracy: [0.5217391304347826, 0.5043478260869565, 0.46956521739130436, 0.5652173913043478, 0.5043478260869565]
Train-wise accuracy: [0.5652173913043478, 0.5130434782608696, 0.5043478260869565, 0.43478260869565216]
Discount % RMSE: 8.845393698703218
Discounted Price RMSE: 104.12050689278283

Prediction for 14-06-2020:
Item_B on Eurostar: 28% off, now €342.17
Item_C on Thalys: 30% off, now €194.37


In [4]:
import joblib
joblib.dump(xgb_item, "xgb_item.pkl")
joblib.dump(xgb_train, "xgb_train.pkl")
joblib.dump(xgb_reg_discount, "xgb_reg_discount.pkl")
joblib.dump(xgb_reg_price, "xgb_reg_price.pkl")
joblib.dump(mlb_item, "mlb_item.pkl")
joblib.dump(mlb_train, "mlb_train.pkl")
print("Successfully for 6 files")

Successfully for 6 files


In [5]:
pip install catboost lightgbm

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
    --------------------------------------- 1.8/100.2 MB 11.9 MB/s eta 0:00:09
   - -------------------------------------- 4.5/100.2 MB 11.9 MB/s eta 0:00:09
   -- ------------------------------------- 6.8/100.2 MB 11.8 MB/s eta 0:00:08
   --- ------------------------------------ 9.4/100.2 MB 11.9 MB/s eta 0:00:08
   ---- ----------------------------------- 11.8/100.2 MB 11.8 MB/s eta 0:00:08
   ----- ---------------------------------- 14.2/100.2 MB 11.8 MB/s eta 0:00:08
   ------ --------------------------------- 16.5/100.2 MB 11.8 MB/s eta 0:00:08
   ------- -------------------------------- 19.1/100.2 MB 11.8 MB/s eta 0:00:07
   -------- ------------------------------- 21.5/100.2 MB 11.8 MB/s eta 0:00:07
   --------- ------------------------------ 24.1/100.2 MB 11.8 MB/s eta 0:00:07
   ---------- ----------------------------- 26.5/100.2 MB 11.8 MB/s eta 0:00:07
   ----------- ---------------------------- 28.8/100.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
